Task 1

In [17]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]


def clean_machakos(raw, month_names):
    valid_values, valid_flags = [], []
    for entry in raw:
        try:
            valid_values.append(int(str(entry).strip()))
            valid_flags.append(True)
        except ValueError:
            valid_flags.append(False)

    impute_value = round(sum(valid_values) / len(valid_values))
    cleaned, imputed_months = [], []
    for i, entry in enumerate(raw):
        if valid_flags[i]:
            cleaned.append(int(str(entry).strip()))
        else:
            cleaned.append(impute_value)
            imputed_months.append(month_names[i])
    return cleaned, imputed_months


machakos_clean, imputed_months = clean_machakos(machakos_raw, MONTHS)

print("Imputed months:", imputed_months)
print("Cleaned Machakos:", machakos_clean)

Imputed months: ['April', 'September']
Cleaned Machakos: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


Task 2

In [18]:
R = np.array(clean_rows + [machakos_clean], dtype=np.int64)

print("Shape:", R.shape)
print("Dtype:", R.dtype)

Shape: (5, 12)
Dtype: int64


Task 3

In [19]:
annual_totals   = R.sum(axis=1)     # per county
monthly_averages = R.mean(axis=0)   # per month across counties

print("Annual totals (mm):")
for c, t in zip(counties, annual_totals):
    print(f"  {c:<10} {t}")

print("\nMonthly averages across all counties (mm):")
for m, a in zip(MONTHS, monthly_averages):
    print(f"  {m:<10} {a:.1f}")

Annual totals (mm):
  Nairobi    1045
  Nakuru     916
  Kisumu     1371
  Mombasa    1075
  Machakos   811

Monthly averages across all counties (mm):
  January    47.0
  February   47.6
  March      91.0
  April      164.4
  May        171.2
  June       67.6
  July       56.6
  August     58.2
  September  64.2
  October    67.8
  November   127.4
  December   80.6


Task 4

In [20]:
wettest_idx    = np.argmax(R, axis=1)
wettest_months = [MONTHS[i] for i in wettest_idx]

for county, month in zip(counties, wettest_months):
    print(f"{county:<10} -> {month}")

Nairobi    -> April
Nakuru     -> April
Kisumu     -> April
Mombasa    -> May
Machakos   -> November


Task 5

In [21]:
dry_mask   = R < 50
dry_months = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_months):
    print(f"{county:<10} -> {n} dry month(s)")

Nairobi    -> 4 dry month(s)
Nakuru     -> 3 dry month(s)
Kisumu     -> 0 dry month(s)
Mombasa    -> 2 dry month(s)
Machakos   -> 6 dry month(s)


Task 6

In [22]:
pair_sums  = R[:, :-1] + R[:, 1:]          # 11 consecutive 2-month sums
best_start = np.argmax(pair_sums, axis=1)  # start column of best pair

planting_windows = []
for i, s in enumerate(best_start):
    window = f"{MONTHS[s]}-{MONTHS[s+1]}"
    planting_windows.append(window)
    print(f"{counties[i]:<10} -> {window}  ({pair_sums[i, s]} mm combined)")

Nairobi    -> April-May  (395 mm combined)
Nakuru     -> April-May  (265 mm combined)
Kisumu     -> April-May  (420 mm combined)
Mombasa    -> April-May  (390 mm combined)
Machakos   -> November-December  (278 mm combined)


Task 7

In [23]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month",
                     "dry_months", "planting_window"])
    for i, c in enumerate(counties):
        writer.writerow([c, int(annual_totals[i]), wettest_months[i],
                         int(dry_months[i]), planting_windows[i]])

print(open("rainfall_summary.csv").read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December

